# Custom sets and dataset discovery

**Goal.** Build a small immutable user-provided scalar set, use it as a
policy base, and inspect metadata for packaged alternatives.

**Prerequisites.** Familiarity with element symbols and basic scalar
lookups is sufficient. The example values are illustrative covalent radii
in angstrom, not a new curated dataset shipped by `atomref`.

## Setup

Import the public namespace used to construct sets, policies, and dataset
references.

In [1]:
import atomref as ar


## Define a custom base set

The custom set supplies C and O directly. An explicit substitution step
uses the packaged Cordero table for N, so each lookup retains whether it
came from the custom base or the fallback source.

In [2]:
custom_cov = ar.ElementScalarSet.from_mapping(
    ref=ar.DatasetRef("covalent_radius", "demo_user_cov"),
    values={"C": 0.77, "O": 0.67},
    name="Demo user covalent set",
    units="angstrom",
    description="Example custom set for notebook usage.",
    notes=("Notebook example",),
)

policy = ar.RadiiPolicy(
    kind="covalent",
    base_set=custom_cov,
    transfers=(
        ar.SubstitutionTransfer(
            source=ar.DatasetRef("covalent_radius", "cordero2008")
        ),
    ),
)

for symbol in ("C", "O", "N"):
    print(symbol, ar.lookup_covalent_radius(symbol, policy=policy))


C LookupResult(value=0.77, source='base', target=DatasetRef(quantity='covalent_radius', set_id='demo_user_cov'), resolved_from=(DatasetRef(quantity='covalent_radius', set_id='demo_user_cov'),), is_placeholder=False, fit=None, notes=(), transfer_depth=0)
O LookupResult(value=0.67, source='base', target=DatasetRef(quantity='covalent_radius', set_id='demo_user_cov'), resolved_from=(DatasetRef(quantity='covalent_radius', set_id='demo_user_cov'),), is_placeholder=False, fit=None, notes=(), transfer_depth=0)
N LookupResult(value=0.71, source='transfer_substitution', target=DatasetRef(quantity='covalent_radius', set_id='demo_user_cov'), resolved_from=(DatasetRef(quantity='covalent_radius', set_id='cordero2008'),), is_placeholder=False, fit=None, notes=('missing in base set; substituted from transfer source',), transfer_depth=1)


## Discover packaged dataset metadata

List target van der Waals tables and inspect the scientific classification
and usage role of the Rahm atomic-radius support dataset.

In [3]:
for info in ar.list_radii_set_infos("van_der_waals", usage_role="target"):
    print(info.ref.set_id, info.semantic_class, info.origin_class, info.phase_context)

rahm = ar.get_dataset_info(ar.DatasetRef("atomic_radius", "rahm2016"))
print(rahm.name)
print(rahm.semantic_class, rahm.phase_context, rahm.usage_role)


bondi1964 vdw_compiled compiled_experimental mixed_or_legacy
rowland_taylor1996 vdw_structural structural condensed_phase
alvarez2013 vdw_structural structural condensed_phase
chernyshov2020 vdw_structural_typed_reduced structural condensed_phase
Rahm isodensity atomic radii (ρ=0.001 e/bohr³)
atomic_isodensity isolated_atom support


## What this demonstrated and limitations

A custom `ElementScalarSet` can participate in the same scalar policy
machinery as packaged tables without modifying the global registry. Users
remain responsible for the provenance, units, coverage, and scientific
fitness of their own values. Custom radial-profile datasets are outside the
current public scope.